In [2]:
# ! pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client pandas

In [3]:
import pandas as pd
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
import os.path
import pickle

# If modifying these scopes, delete the file token.pickle.
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']

def get_credentials(credentials_path):
    """Gets valid user credentials from storage."""
    creds = None
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                credentials_path, 
                SCOPES,
                redirect_uri='http://localhost:8080'  # Specify the redirect URI
            )
            # Run the local server at port 8080
            creds = flow.run_local_server(port=8080)
            
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)
    
    return creds

def get_google_sheet_data(SPREADSHEET_ID, RANGE_NAME, credentials_path):
    """Fetch data from Google Sheets using OAuth 2.0 credentials"""
    try:
        creds = get_credentials(credentials_path)
        service = build('sheets', 'v4', credentials=creds)
        sheet = service.spreadsheets()
        result = sheet.values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=RANGE_NAME
        ).execute()

        values = result.get('values', [])
        if not values:
            print('No data found.')
            return None

        df = pd.DataFrame(values[1:], columns=values[0])
        return df

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Example usage
if __name__ == "__main__":
    SPREADSHEET_ID = '1TNOCQb-YTGDwWAqDrw9IJlyu7Ng9w2o6F65LwlANOiU'
    RANGE_NAME = 'Sheet1!A1:E100'
    CREDENTIALS_PATH = 'client_secret_378226750851-c4k4anukqpjf5cp4q9u9gbqr38nuqv4q.apps.googleusercontent.com.json'

    df = get_google_sheet_data(SPREADSHEET_ID, RANGE_NAME, CREDENTIALS_PATH)
    
    if df is not None:
        print("Data fetched successfully:")
        print(df)


An error occurred: [Errno 48] Address already in use
